# Banking Intent Classifier — Day 2: Build a Text Baseline

**Goal:** prepare the data for fair model experiments, then build baselines and one small LSTM.
**Dataset:** PolyAI Banking77 (77 banking intents). Same task as Day 1.

**Notebook structure**
0. Setup
1. Part 1 — Agree on the experiment (split, columns, metric, seed, label encoding)
2. Part 2 — Work in parallel
   - Track A — Prepare the text (tokenization, vocabulary, padding, leakage checks)
   - Track B — Baseline models (dummy, TF-IDF + Logistic Regression, cross-validation, LSTM)
3. Part 3 — Afternoon checkpoint: integrate everything into one pipeline (final validation, test evaluation, prediction)
4. Day 2 summary

**Golden rules for today**
- Anything learned from data (vocabulary, max length, label classes) is fitted on TRAIN only.
- Validation is for choices. The test set is touched ONCE, at the very end.
- Every model uses the same split, the same label encoding, and the same evaluation function.

## 0. Setup

In [ ]:
import re
import copy
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.metrics.pairwise import linear_kernel

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

set_seed()

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

---
# Part 1 — Agree on the experiment (do this together, BEFORE modeling)

| Decision | Our choice |
|---|---|
| Task | Classify a customer banking query into one of 77 intents using its text |
| Text column / label column | `text` / `category` |
| Train / validation / test | Official PolyAI `train.csv` is split 80% train / 20% validation (stratified, seed 42). Official `test.csv` is the test set and stays untouched until the final evaluation |
| Related examples (author, customer, conversation) | The dataset has no such column, so nothing needs to be grouped. Stratification is therefore safe |
| Main metric | **Macro F1** (every intent counts equally) |
| Supporting metrics | Accuracy, Weighted F1, Top-3 accuracy |
| Seed | `SEED = 42` everywhere |
| Label encoding | One `LabelEncoder` fitted on train labels, shared by all models |
| Evaluation function | One `evaluate()` function used by all models |
| Text preparation | One shared preprocessing (Track A): lowercase, keep punctuation, keep stopwords |
| Dataset size | ~10k train rows, small enough to use fully (no subset needed) |

### 1.1 Load the data and create train / validation / test

In [ ]:
train_url = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/train.csv"
test_url = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/test.csv"

TEXT_COL = "text"
LABEL_COL = "category"

full_train_df = pd.read_csv(train_url)
test_df = pd.read_csv(test_url)

train_df, val_df = train_test_split(
    full_train_df,
    test_size=0.2,
    stratify=full_train_df[LABEL_COL],
    random_state=SEED,
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Original train file:", full_train_df.shape)
print("Train      :", train_df.shape)
print("Validation :", val_df.shape)
print("Test       :", test_df.shape, "(untouched until the final evaluation)")

### 1.2 Document the split and class counts

In [ ]:
split_counts = pd.DataFrame({
    "train": train_df[LABEL_COL].value_counts(),
    "validation": val_df[LABEL_COL].value_counts(),
    "test": test_df[LABEL_COL].value_counts(),
}).fillna(0).astype(int)

summary = pd.DataFrame({
    "rows": split_counts.sum(),
    "classes": (split_counts > 0).sum(),
    "min per class": split_counts.min(),
    "max per class": split_counts.max(),
})
display(summary)

print("Smallest classes (train / validation / test):")
display(split_counts.sort_values("train").head(5))

### 1.3 One label encoding and one evaluation function (shared by every model)

In [ ]:
label_encoder = LabelEncoder().fit(train_df[LABEL_COL])
CLASSES = label_encoder.classes_
N_CLASSES = len(CLASSES)

y_train = label_encoder.transform(train_df[LABEL_COL])
y_val = label_encoder.transform(val_df[LABEL_COL])
y_test = label_encoder.transform(test_df[LABEL_COL])

print("Number of classes:", N_CLASSES)
print("First 5 classes in the shared order:", list(CLASSES[:5]))

# Every model returns probabilities with columns in THIS class order (0..76).
RESULTS = {}

def top_k_accuracy(y_true, proba, k=3):
    top_k = np.argsort(proba, axis=1)[:, -k:]
    return float(np.mean([y in row for y, row in zip(y_true, top_k)]))

def evaluate(y_true, y_pred, name, split, proba=None):
    row = {
        "model": name,
        "split": split,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }
    if proba is not None:
        row["top3_acc"] = top_k_accuracy(y_true, proba, 3)
    RESULTS[(name, split)] = row
    shown = {k: round(v, 4) for k, v in row.items() if k not in ("model", "split")}
    print(f"[{split}] {name}: {shown}")
    return row

def results_table(split=None):
    table = pd.DataFrame(RESULTS.values())
    if split is not None:
        table = table[table["split"] == split]
    return table.round(4).reset_index(drop=True)

---
# Part 2 — Work in parallel

The shared foundation from Part 1 is ready (split, label encoding, metric, seed). Now the work splits in two, each done in pairs:

- **Track A — Prepare the text:** tokenization, vocabulary size, maximum length, padding, label encoding, leakage checks.
- **Track B — Baseline models, evaluation, and prediction:** dummy baseline, simple model, cross-validation, then one small LSTM.

Track A hands a working preprocessing to Track B at the checkpoint (Part 3).

## Track A — Prepare the text

Goals: agree on tokenization, vocabulary size, maximum sequence length, padding and label encoding. Inspect tokenized examples. Fit everything data-based on **training text only**. Keep the preparation reusable (functions and one class).

| Choice | Decision | Why |
|---|---|---|
| Normalization | lowercase + collapse whitespace | Day 1: keep punctuation and stopwords, they carry meaning (e.g. "not", "?") |
| Tokenization | words (with apostrophes) and each punctuation mark as its own token | simple and reproducible |
| Vocabulary size | max 5,000 tokens, from train only. Index 0 = `<PAD>`, index 1 = `<OOV>` | Banking77 has a small vocabulary |
| Maximum length | 95th percentile of train token lengths | covers almost every text without long padding |
| Padding / truncation | both at the END (`post`) | keeps the start of each message |
| Label encoding | `LabelEncoder` from Part 1 | shared by all models |

### A1. Reusable preprocessing functions and class

In [ ]:
TOKEN_PATTERN = re.compile(r"[a-z0-9]+(?:'[a-z]+)?|[^\sa-z0-9]")
PAD_TOKEN, OOV_TOKEN = "<PAD>", "<OOV>"
PAD_ID, OOV_ID = 0, 1

def clean_text(text):
    """Minimal cleaning: lowercase, trim, collapse repeated whitespace."""
    text = str(text).lower().strip()
    return re.sub(r"\s+", " ", text)

def tokenize(text):
    """Split a text into word tokens and punctuation tokens."""
    return TOKEN_PATTERN.findall(clean_text(text))

class TextPreprocessor:
    """Text -> padded integer matrix.

    fit(texts)        learns the vocabulary and max length (call with TRAIN text only)
    transform(texts)  returns an int32 array of shape (n_texts, max_len_)
    """

    def __init__(self, max_vocab=5000, max_len=None, len_percentile=95):
        self.max_vocab = max_vocab
        self.max_len = max_len
        self.len_percentile = len_percentile

    def tokens(self, text):
        """The ONE place where text becomes tokens. Override this to change the preparation."""
        return tokenize(text)

    def fit(self, texts):
        tokenized = [self.tokens(t) for t in texts]
        counts = Counter(tok for toks in tokenized for tok in toks)
        kept = [tok for tok, _ in counts.most_common(self.max_vocab - 2)]
        self.itos = [PAD_TOKEN, OOV_TOKEN] + kept
        self.stoi = {tok: i for i, tok in enumerate(self.itos)}
        if self.max_len is None:
            lengths = [len(toks) for toks in tokenized]
            self.max_len_ = int(np.percentile(lengths, self.len_percentile))
        else:
            self.max_len_ = int(self.max_len)
        return self

    @property
    def vocab_size(self):
        return len(self.itos)

    def encode(self, text):
        return [self.stoi.get(tok, OOV_ID) for tok in self.tokens(text)]

    def decode(self, ids):
        return [self.itos[i] for i in ids if i != PAD_ID]

    def transform(self, texts):
        texts = list(texts)
        X = np.zeros((len(texts), self.max_len_), dtype="int32")
        for row, text in enumerate(texts):
            ids = self.encode(text)[: self.max_len_]
            X[row, : len(ids)] = ids
        return X

    def fit_transform(self, texts):
        return self.fit(texts).transform(texts)

### A2. 🔌 PREPROCESSING INTEGRATION POINT

This is the **only place** where the preprocessing is created and fitted. Track A hands this object to Track B.
To swap in a more advanced preprocessing later, replace the `preprocessor = ...` line with any object that has:
`.fit(texts)`, `.transform(texts)` returning an int array of shape `(n, max_len)`, plus `.vocab_size` and `.max_len_`.

In [ ]:
# 🔌 PREPROCESSING INTEGRATION POINT -----------------------------------------
preprocessor = TextPreprocessor(max_vocab=5000, len_percentile=95)
preprocessor.fit(train_df[TEXT_COL])          # TRAIN text only
# ----------------------------------------------------------------------------

X_train = preprocessor.transform(train_df[TEXT_COL])
X_val = preprocessor.transform(val_df[TEXT_COL])
X_test = preprocessor.transform(test_df[TEXT_COL])

train_lengths = np.array([len(preprocessor.tokens(t)) for t in train_df[TEXT_COL]])
print("Vocabulary size (with PAD and OOV):", preprocessor.vocab_size)
print("Max sequence length chosen:", preprocessor.max_len_)
print("Train token length: mean =", round(train_lengths.mean(), 1),
      "| 95th percentile =", int(np.percentile(train_lengths, 95)),
      "| max =", train_lengths.max())
print("Train texts that get truncated:",
      f"{(train_lengths > preprocessor.max_len_).mean():.1%}")
print("Shapes:", X_train.shape, X_val.shape, X_test.shape)

### A3. Inspect tokenized examples and check shapes and labels

In [ ]:
for i in train_df.sample(4, random_state=SEED).index:
    print("Text   :", train_df.loc[i, TEXT_COL])
    print("Tokens :", preprocessor.tokens(train_df.loc[i, TEXT_COL]))
    print("IDs    :", X_train[i].tolist())
    print("Decoded:", preprocessor.decode(X_train[i]))
    print("Label  :", y_train[i], "->", CLASSES[y_train[i]])
    print()

# Automatic checks
assert X_train.shape == (len(train_df), preprocessor.max_len_)
assert X_val.shape == (len(val_df), preprocessor.max_len_)
assert X_test.shape == (len(test_df), preprocessor.max_len_)
assert len(X_train) == len(y_train) == len(train_df)
assert (label_encoder.inverse_transform(y_train) == train_df[LABEL_COL].values).all()
assert (label_encoder.inverse_transform(y_val) == val_df[LABEL_COL].values).all()
assert X_train.max() < preprocessor.vocab_size and X_train.min() >= 0
print("All shape and label checks passed.")

### A4. Data leakage checks

Questions: are similar examples in train and validation/test? Do the same authors come back? Is anything identifying in the text?
Vocabulary and max length were fitted on train only, so validation/test words that are not in train become `<OOV>`.

In [ ]:
# 1) Out-of-vocabulary rate (should be higher on val/test than on train, because the vocabulary is train-only)
def oov_rate(texts):
    tokens = [tok for t in texts for tok in preprocessor.tokens(t)]
    return sum(tok not in preprocessor.stoi for tok in tokens) / len(tokens)

print("OOV rate  train: {:.2%} | validation: {:.2%} | test: {:.2%}".format(
    oov_rate(train_df[TEXT_COL]), oov_rate(val_df[TEXT_COL]), oov_rate(test_df[TEXT_COL])))

# 2) Exact duplicates AFTER normalization, inside and across splits
norm = {
    "train": set(train_df[TEXT_COL].map(clean_text)),
    "validation": set(val_df[TEXT_COL].map(clean_text)),
    "test": set(test_df[TEXT_COL].map(clean_text)),
}
print("\nDuplicates inside train after cleaning:",
      len(train_df) - len(norm["train"]))
print("Shared train/validation:", len(norm["train"] & norm["validation"]))
print("Shared train/test      :", len(norm["train"] & norm["test"]))
print("Shared validation/test :", len(norm["validation"] & norm["test"]))

In [ ]:
# 3) Near-duplicates: how similar is each validation/test text to its closest TRAIN text?
sim_vectorizer = TfidfVectorizer(tokenizer=tokenize, token_pattern=None, lowercase=False)
X_train_sim = sim_vectorizer.fit_transform(train_df[TEXT_COL])   # fitted on train only

def max_similarity(texts, chunk=500):
    X_other = sim_vectorizer.transform(texts)
    best = []
    for start in range(0, X_other.shape[0], chunk):
        sims = linear_kernel(X_other[start:start + chunk], X_train_sim)
        best.append(sims.max(axis=1))
    return np.concatenate(best)

for name, df in [("validation", val_df), ("test", test_df)]:
    sims = max_similarity(df[TEXT_COL])
    print(f"{name}: similarity >= 0.95 with a train text: {(sims >= 0.95).sum()} | "
          f">= 0.80: {(sims >= 0.80).sum()} | median similarity: {np.median(sims):.2f}")

# 4) Authors / identifying elements
patterns = {
    "email-like": r"[\w\.-]+@[\w\.-]+\.\w+",
    "long digit sequences (7+)": r"\d{7,}",
    "URLs": r"http|www\.",
}
for name, pat in patterns.items():
    total = sum(df[TEXT_COL].str.contains(pat, case=False, regex=True).sum()
                for df in (train_df, val_df, test_df))
    print(f"Texts with {name}: {total}")

### Track A notes (leakage and decisions)

- **Fit on train only:** vocabulary, max length and label classes come from the training split. Validation and test are only transformed.
- **Authors / related examples:** no author, customer or conversation column exists, so they cannot be grouped. We rely on the official train/test split.
- **Exact duplicates across splits:** __ (fill from A4).
- **Near-duplicates:** some validation/test texts are paraphrases of train texts (similarity >= 0.80: __). This is normal in Banking77 because each intent has many paraphrases. We keep them and mention it when reporting scores.
- **Identifying information:** none found in the checks above (fill the counts).
- **Handover to Track B:** `clean_text`, `tokenize`, `TextPreprocessor`, the fitted `preprocessor`, and the arrays `X_train`, `X_val`, `X_test`.

---
## Track B — Baseline models, evaluation, and prediction

Order for today: (1) dummy baseline, (2) simple model, (3) cross-validation, (4) record the scores, (5) one short LSTM run to prove the pipeline works.
The test set is NOT used in this section.

### B1. Dummy baseline (always predicts the most common class)

This is the minimum result for the day. Any real model must beat it.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(np.zeros((len(y_train), 1)), y_train)

dummy_val_proba = dummy.predict_proba(np.zeros((len(y_val), 1)))
dummy_val_pred = dummy.predict(np.zeros((len(y_val), 1)))

print("Always predicts:", CLASSES[dummy_val_pred[0]])
evaluate(y_val, dummy_val_pred, "Dummy (most frequent)", "validation", dummy_val_proba)

### B2. Simple model: TF-IDF + Logistic Regression

Uses the same `tokenize` function as the shared preprocessing. The TF-IDF vocabulary is learned inside the pipeline, from the training texts only.

In [ ]:
tfidf_lr = Pipeline([
    ("tfidf", TfidfVectorizer(tokenizer=tokenize, token_pattern=None, lowercase=False,
                              ngram_range=(1, 2), sublinear_tf=True)),
    ("clf", LogisticRegression(C=10, max_iter=500)),
])

tfidf_lr.fit(train_df[TEXT_COL], y_train)

tfidf_val_proba = tfidf_lr.predict_proba(val_df[TEXT_COL])
tfidf_val_pred = tfidf_val_proba.argmax(axis=1)
evaluate(y_val, tfidf_val_pred, "TF-IDF + LogReg", "validation", tfidf_val_proba)

### B3. Cross-validated evaluation (on the training split only)

The vectorizer is inside the pipeline, so it is re-fitted inside each fold. That avoids leakage between folds.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

cv_dummy = cross_val_score(
    DummyClassifier(strategy="most_frequent"),
    np.zeros((len(y_train), 1)), y_train, cv=cv, scoring="f1_macro")

cv_tfidf = cross_val_score(
    tfidf_lr, train_df[TEXT_COL], y_train, cv=cv, scoring="f1_macro", n_jobs=-1)

print(f"Dummy           5-fold Macro F1: {cv_dummy.mean():.4f} ± {cv_dummy.std():.4f}")
print(f"TF-IDF + LogReg 5-fold Macro F1: {cv_tfidf.mean():.4f} ± {cv_tfidf.std():.4f}")
print("Per-fold scores (TF-IDF):", cv_tfidf.round(4))

### B4. Record the baseline scores BEFORE training the LSTM

In [ ]:
display(results_table("validation"))

### B5. One short LSTM run (proves the pipeline works)

Embedding layer + padded sequences + one LSTM layer. Only 3 epochs here: the goal is to see the loss go down and the shapes line up, not to get the best score. The full training happens in Part 3.

In [ ]:
def build_lstm(vocab_size, max_len, n_classes, embed_dim=100, lstm_units=128, dropout=0.3):
    model = keras.Sequential([
        layers.Input(shape=(max_len,)),
        layers.Embedding(vocab_size, embed_dim, mask_zero=True),
        layers.LSTM(lstm_units),
        layers.Dropout(dropout),
        layers.Dense(n_classes, activation="softmax"),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

set_seed()
quick_model = build_lstm(preprocessor.vocab_size, preprocessor.max_len_, N_CLASSES)
quick_model.summary()

print("Input shape:", X_train.shape, "| labels:", y_train.shape)
quick_history = quick_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=3, batch_size=64, verbose=2,
)

---
# Part 3 — Afternoon checkpoint: bring the work together

Track A hands over the preprocessing; Track B wraps it with the model into **one object** that goes from raw text to a predicted intent.
To replace the preprocessing later, change only the `preprocessor` in the integration point (A2), then re-run from here.

### 3.1 Full pipeline: preprocessing → LSTM → label decoding

In [ ]:
class TextClassifierPipeline:
    """raw text -> preprocessor -> model -> intent name."""

    def __init__(self, preprocessor, build_model, label_encoder,
                 epochs=30, batch_size=64, patience=5):
        self.preprocessor = preprocessor
        self.build_model = build_model
        self.label_encoder = label_encoder
        self.epochs = epochs
        self.batch_size = batch_size
        self.patience = patience

    def fit(self, train_texts, y_train_enc, val_texts, y_val_enc):
        set_seed()
        self.preprocessor.fit(train_texts)                    # TRAIN text only
        X_tr = self.preprocessor.transform(train_texts)
        X_va = self.preprocessor.transform(val_texts)

        self.model = self.build_model(
            self.preprocessor.vocab_size, self.preprocessor.max_len_,
            len(self.label_encoder.classes_))

        stop = keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=self.patience, restore_best_weights=True)
        self.history = self.model.fit(
            X_tr, y_train_enc,
            validation_data=(X_va, y_val_enc),
            epochs=self.epochs, batch_size=self.batch_size,
            callbacks=[stop], verbose=2)
        return self

    def predict_proba(self, texts):
        return self.model.predict(self.preprocessor.transform(texts), verbose=0)

    def predict(self, texts):
        return self.predict_proba(texts).argmax(axis=1)

    def predict_intent(self, texts, top_k=3):
        proba = self.predict_proba(texts)
        results = []
        for row in proba:
            top = row.argsort()[::-1][:top_k]
            results.append([(self.label_encoder.classes_[i], float(row[i])) for i in top])
        return results

In [ ]:
pipeline = TextClassifierPipeline(preprocessor, build_lstm, label_encoder)
pipeline.fit(train_df[TEXT_COL], y_train, val_df[TEXT_COL], y_val)

### 3.2 Training curves

In [ ]:
hist = pipeline.history.history
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(hist["loss"], label="train")
axes[0].plot(hist["val_loss"], label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(hist["accuracy"], label="train")
axes[1].plot(hist["val_accuracy"], label="validation")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

### 3.3 Final validation of the whole pipeline (validation set)

In [ ]:
lstm_val_proba = pipeline.predict_proba(val_df[TEXT_COL])
lstm_val_pred = lstm_val_proba.argmax(axis=1)
evaluate(y_val, lstm_val_pred, "LSTM pipeline", "validation", lstm_val_proba)

print("\nAll models on the validation set:")
display(results_table("validation"))

report = classification_report(
    y_val, lstm_val_pred, labels=range(N_CLASSES), target_names=CLASSES,
    output_dict=True, zero_division=0)
report_df = (pd.DataFrame(report).T
             .drop(["accuracy", "macro avg", "weighted avg"])
             .sort_values("f1-score"))
print("\nWeakest 10 intents for the LSTM pipeline (validation):")
display(report_df.head(10).round(3))

### 3.4 Final evaluation on the TEST set (run once)

⚠️ Do not go back and change the models after seeing these numbers. If you do, the test set is no longer a fair final check.

In [ ]:
# Dummy
dummy_test_proba = dummy.predict_proba(np.zeros((len(y_test), 1)))
evaluate(y_test, dummy_test_proba.argmax(axis=1), "Dummy (most frequent)", "test", dummy_test_proba)

# TF-IDF + LogReg
tfidf_test_proba = tfidf_lr.predict_proba(test_df[TEXT_COL])
evaluate(y_test, tfidf_test_proba.argmax(axis=1), "TF-IDF + LogReg", "test", tfidf_test_proba)

# LSTM pipeline (raw text in, everything else inside the pipeline)
lstm_test_proba = pipeline.predict_proba(test_df[TEXT_COL])
evaluate(y_test, lstm_test_proba.argmax(axis=1), "LSTM pipeline", "test", lstm_test_proba)

print("\nFinal comparison on the test set:")
display(results_table("test"))

### 3.5 Prediction on new data

In [ ]:
new_texts = [
    "I lost my card yesterday, please block it",
    "Why was my transfer declined?",
    "How long does a top up take?",
    "My card has not arrived yet",
]

lstm_preds = pipeline.predict_intent(new_texts, top_k=3)
tfidf_new_proba = tfidf_lr.predict_proba(new_texts)

for text, preds, tf_row in zip(new_texts, lstm_preds, tfidf_new_proba):
    print("Text:", text)
    for label, prob in preds:
        print(f"   LSTM   {label:<40} {prob:.1%}")
    best = tf_row.argmax()
    print(f"   TF-IDF {CLASSES[best]:<40} {tf_row[best]:.1%}\n")

### 3.6 How to swap the preprocessing later

Example: a version that is smarter about numbers. Write a new class (or a subclass), then change ONLY the `preprocessor = ...` line in A2 and re-run from A2 downward.

```python
class AdvancedPreprocessor(TextPreprocessor):
    def tokens(self, text):                         # used by fit, encode and the length statistics
        text = re.sub(r"\d+", "0", str(text))      # example: every number becomes "0"
        return tokenize(text)

preprocessor = AdvancedPreprocessor(max_vocab=5000, len_percentile=95)
preprocessor.fit(train_df[TEXT_COL])
```
Because `fit`, `encode` and the OOV/length checks all call `tokens()`, one override changes everything consistently.

---
# Day 2 Summary

**Experiment:** 77-intent classification, 80/20 stratified train/validation split of the official train file, official test file untouched until the end. Seed 42. Main metric: Macro F1.

**Scores (fill from the tables above)**

| Model | Validation Macro F1 | Test Macro F1 | Test Accuracy |
|---|---|---|---|
| Dummy (most frequent) | __ | __ | __ |
| TF-IDF + Logistic Regression | __ | __ | __ |
| LSTM pipeline | __ | __ | __ |

**Cross-validation (TF-IDF, 5-fold Macro F1):** __ ± __

**Leakage checks:** vocabulary, max length and label classes fitted on train only; exact duplicates across splits: __; near-duplicates are normal paraphrases; no authors or identifying text.

**What worked / what was hard:** __

**Next steps:** compare with a pretrained embedding or a fine-tuned transformer, and look at the confused intent pairs.